In [9]:
# Load the matched occupational dataset for analysis.
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import numpy as np

ROOT = Path("/content/drive/MyDrive/GenAI-human-skill-adaptation-dev/Occupational-level")

analysis_data = pd.read_csv(
    ROOT / "04_Analysis_Dataset" / "Output" / "Occupational_Analysis_Dataset.csv"
)

In [10]:
# Define analysis variables and count occupations by sector-assignment pattern.
outcome = "augmentation_share"

# Define the five skill bundles using the agreed descriptive names.
bundle_cols = [
    "Technical and analytical skills",
    "Critical thinking and evaluation",
    "Communication and interpersonal skills",
    "Complex problem solving and systems analysis",
    "Learning and time management",
]

sector_cols = [
    "sector_finance",
    "sector_healthcare",
    "sector_education",
    "sector_legal",
]

### Which bundles are associated with augmentation share, and how large are those associations?

Main

In [11]:
# Main — Results: estimate joint associations and their magnitude.
model_data = analysis_data[[outcome] + bundle_cols + sector_cols].astype(float)
model_data[bundle_cols] = (
    model_data[bundle_cols] - model_data[bundle_cols].mean()
) / model_data[bundle_cols].std(ddof=1)

joint_model = sm.GLM(
    model_data[outcome],
    sm.add_constant(model_data[bundle_cols + sector_cols]),
    family=sm.families.Binomial()
).fit(cov_type="HC0")

assert joint_model.converged, "Joint model did not converge."

# Express associations as average marginal effects in percentage points.
ame = joint_model.get_margeff(at="overall", method="dydx")
ci = ame.conf_int()

ame_results = pd.DataFrame({
    "Average marginal effect": ame.margeff * 100,
    "95% CI lower": ci[:, 0] * 100,
    "95% CI upper": ci[:, 1] * 100,
    "p": ame.pvalues
}, index=joint_model.model.exog_names[1:]).loc[bundle_cols].rename_axis("Bundle")

ame_results["p (Holm)"] = multipletests(
    ame_results.pop("p"), method="holm"
)[1]

# Display the primary results using consistent labels and precision.
p_format = lambda p: "< 0.001" if p < 0.001 else f"{p:.3f}"

In [12]:
display(
    ame_results.style
    .set_caption(
        "Main — Results: Skill-bundle associations with "
        f"GenAI augmentation share (N = {int(joint_model.nobs)})"
    )
    .format({"p (Holm)": p_format}, precision=3)
)

,Average marginal effect,95% CI lower,95% CI upper,p (Holm)
Bundle,,,,
Technical and analytical skills,6.639,4.394,8.885,< 0.001
Critical thinking and evaluation,-12.159,-15.299,-9.020,< 0.001
Communication and interpersonal skills,8.210,5.431,10.988,< 0.001
Complex problem solving and systems analysis,13.257,9.586,16.928,< 0.001
Learning and time management,0.622,-2.221,3.464,0.668


### Do these findings persist after accounting for broad occupational groups?

In [13]:
# Identify SOC3 groups and the comparison sample, excluding singleton groups.
soc3 = (
    analysis_data.loc[joint_model.model.data.row_labels, "soc6"]
    .astype("string")
    .str.replace("-", "", regex=False)
    .str[:3]
)
soc3_counts = soc3.value_counts()
soc3_keep = soc3.map(soc3_counts).gt(1)
print(f"SOC3 groups: {soc3.nunique()}")
print(f"Singleton occupations: {(~soc3_keep).sum()}")
print(f"Comparison sample: {soc3_keep.sum()} of {len(soc3)} occupations")

SOC3 groups: 89
Singleton occupations: 6
Comparison sample: 579 of 585 occupations


In [14]:
# Fit joint models with and without SOC3 fixed effects on the same occupations.
matched_data = model_data.loc[soc3.index[soc3_keep]]
matched_X = sm.add_constant(matched_data[bundle_cols + sector_cols])
soc3_X = matched_X.join(
    pd.get_dummies(
        soc3.loc[matched_data.index],
        prefix="SOC3", drop_first=True, dtype=float
    )
)
soc3_models, soc3_tables = {}, {}

for label, X in {
    "Without SOC3 FE": matched_X,
    "With SOC3 FE": soc3_X
}.items():
    assert np.linalg.matrix_rank(X) == X.shape[1], \
        f"{label}: predictors are not full rank."

    model = sm.GLM(
        matched_data[outcome], X, family=sm.families.Binomial()
    ).fit(cov_type="HC0")

    assert model.converged, f"{label}: model did not converge."
    soc3_models[label] = model

    # Calculate marginal effects and confidence limits in percentage points.
    margins = model.get_margeff(at="overall", method="dydx")
    ci = margins.conf_int()

    table = pd.DataFrame({
        "Marginal effect": margins.margeff * 100,
        "95% CI lower": ci[:, 0] * 100,
        "95% CI upper": ci[:, 1] * 100,
        "p": margins.pvalues
    }, index=X.columns[1:]).loc[bundle_cols].rename_axis("Bundle")

    table["p"] = multipletests(table["p"], method="holm")[1]
    soc3_tables[label] = table

In [15]:
# Display the matched-sample comparison using consistent table headers.
soc3_comparison = pd.concat(soc3_tables, axis=1)
display(
    soc3_comparison.style
    .set_caption(
        "Skill bundle associations with and without SOC3 fixed effects "
        f"(N = {len(matched_data)})"
    )
    .format(
        {(label, "p"): p_format for label in soc3_tables},
        precision=3
    )
)

### Does removing any single occupation change the findings?

In [16]:
# Check sensitivity of the original joint model to each occupation's exclusion.
import numpy as np

joint_X = joint_model.model.exog
joint_y = joint_model.model.endog
soc6 = analysis_data.loc[joint_model.model.data.row_labels, "soc6"]

bundle_positions = [
    joint_model.model.exog_names.index(bundle) - 1 for bundle in bundle_cols
]

effect_rows, p_rows = [], []
for i in range(len(joint_y)):
    keep = np.arange(len(joint_y)) != i
    refit = sm.GLM(
        joint_y[keep], joint_X[keep], family=sm.families.Binomial()
    ).fit(cov_type="HC0")

    assert refit.converged, f"Refit failed when omitting {soc6.iloc[i]}"

    margins = refit.get_margeff(at="overall", method="dydx")
    effect_rows.append(margins.margeff[bundle_positions] * 100)
    p_rows.append(
        multipletests(margins.pvalues[bundle_positions], method="holm")[1]
    )

# Retain each exclusion's results for the audit.
loo_index = pd.Index(soc6, name="Omitted SOC6")
loo_ame = pd.DataFrame(effect_rows, index=loo_index, columns=bundle_cols)
loo_holm = pd.DataFrame(p_rows, index=loo_index, columns=bundle_cols)

assert np.isfinite(loo_ame.to_numpy()).all()
assert np.isfinite(loo_holm.to_numpy()).all()

# Display the minimum and maximum across all single-occupation exclusions.
exclusion_summary = pd.DataFrame({
    "Marginal effect": [
        f"{lower:.3f} to {upper:.3f}"
        for lower, upper in zip(loo_ame.min(), loo_ame.max())
    ],
    "p (Holm)": [
        "< 0.001" if upper < 0.001
        else f"{p_format(lower)} to {p_format(upper)}"
        for lower, upper in zip(loo_holm.min(), loo_holm.max())
    ]
}, index=bundle_cols).rename_axis("Bundle")

In [17]:
display(
    exclusion_summary.style.set_caption(
        "Skill bundle associations with GenAI augmentation share "
        "after single-occupation exclusions"
    )
)

,Marginal effect,p (Holm)
Bundle,,
Technical and analytical skills,6.297 to 6.875,< 0.001
Critical thinking and evaluation,-12.514 to -11.757,< 0.001
Communication and interpersonal skills,7.979 to 8.528,< 0.001
Complex problem solving and systems analysis,12.910 to 13.706,< 0.001
Learning and time management,0.342 to 0.834,0.564 to 0.811


### What associations appear when each bundle is considered separately?

In [18]:
# Supplement — estimate each bundle's sector-adjusted association separately.
separate_models, separate_rows = {}, []

for bundle in bundle_cols:
    model = sm.GLM(
        model_data[outcome],
        sm.add_constant(model_data[[bundle] + sector_cols]),
        family=sm.families.Binomial()
    ).fit(cov_type="HC0")

    assert model.converged, f"Model did not converge: {bundle}"
    separate_models[bundle] = model

    margins = model.get_margeff(at="overall", method="dydx")
    position = model.model.exog_names.index(bundle) - 1
    lower, upper = margins.conf_int()[position] * 100

    separate_rows.append({
        "Bundle": bundle,
        "Marginal effect": margins.margeff[position] * 100,
        "95% CI lower": lower,
        "95% CI upper": upper,
        "p": margins.pvalues[position]
    })

# Apply Holm adjustment across the five bundle tests and display one table.
separate_ame_results = pd.DataFrame(separate_rows).set_index("Bundle")
separate_ame_results["p"] = multipletests(
    separate_ame_results["p"], method="holm"
)[1]

In [19]:
display(
    separate_ame_results.style
    .set_caption("Skill bundle associations in separate models")
    .format({"p": p_format}, precision=3)
)

,Marginal effect,95% CI lower,95% CI upper,p
Bundle,,,,
Technical and analytical skills,9.764,8.296,11.231,< 0.001
Critical thinking and evaluation,10.537,9.010,12.064,< 0.001
Communication and interpersonal skills,12.769,11.546,13.992,< 0.001
Complex problem solving and systems analysis,13.785,12.494,15.077,< 0.001
Learning and time management,12.598,11.247,13.949,< 0.001
